# R10 · TwinCheck demo for the video (Rathi)

**Runtime → T4 GPU**, switch on **Notebook access** for `GITHUB_TOKEN` and `HF_TOKEN` in the 🔑 panel, then **Run all**.

Steps: 1 setup → 2 tests → 3 run the examples and push them (ends with `PUSHED`; you can skip it) → 4 launch the app (public `gradio.live` link; the cell keeps running while the demo is open, stop it when the recording is done).

Settings in cell 1:
- `VERDICT = "head"` (default): Himasri's trained twin-loss verdict head. Use the **Twin examples** buttons: original (EN) → Contradicted, Arabic twin → Contradicted, unrelated evidence → Unverified. `"rule"` uses the stand-in rule, and the app then shows a red **"Rule stand-in, not the trained model"** banner above every result.
- `FAST = True` (default): rule-based claim splitting + template notes + small spaCy model. Ready in ~5 min. `FAST = False` uses Qwen3-8B for claims and cited notes (first load 15–25 min on a T4).

Every result shows which verdict step made it. Script and shot list: `notes/R10_demo_video_script.md`.


In [ ]:
# 1. Setup
from google.colab import userdata
import os, json
REPO = '/content/twincheck'
if not os.path.exists(REPO):
    !git clone https://github.com/himaallu/Group-Project.git {REPO}
%cd {REPO}
!git config user.name "Rathi"
!git config user.email "335395537+rathiabeth-sudo@users.noreply.github.com"
!git fetch -q origin && git checkout -q main && git pull -q origin main

VERDICT = "head"   # "head" = trained twin-loss verdict head; "rule" = stand-in (labelled on screen)
FAST = True        # True: rule claim split + template notes (~5 min); False: Qwen3-8B (slow first load)

GH = userdata.get('GITHUB_TOKEN').strip()
!git remote set-url --push origin https://{GH}@github.com/himaallu/Group-Project.git
!pip install -q -U gradio rank_bm25 faiss-cpu
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
os.environ['TWINCHECK_NLI'] = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'   # zero-shot NLI
os.environ['TWINCHECK_VERDICT'] = 'models/verdict_twin_s42' if VERDICT == 'head' else 'none'
if FAST:
    !python -m spacy download -q en_core_web_sm
    os.environ.update(TWINCHECK_EXTRACTOR='split', TWINCHECK_EXPLAINER='template',
                      TWINCHECK_SPACY='en_core_web_sm')
else:
    !pip install -q -U bitsandbytes
    !python -m spacy download -q en_core_web_trf
    for k in ('TWINCHECK_EXTRACTOR', 'TWINCHECK_EXPLAINER', 'TWINCHECK_SPACY'):
        os.environ.pop(k, None)
# offline evidence library (~330 MB), only needed when the evidence box is left empty
from huggingface_hub import HfApi
USER = HfApi(token=os.environ['HF_TOKEN']).whoami()['name']
!python -m src.retrieval.build_index download --repo {USER}/twincheck-index-v1
!python -m src.pipeline --status


## 2. Tests

In [ ]:
!python -m pytest tests/demo tests/explain tests/test_pipeline.py -q

## 3. Run the 3 twin examples and the 6 ready examples, push what the model says (optional)

In [ ]:
OUT = f'results/R10/examples_{VERDICT}{"_fast" if FAST else ""}.json'
!python -m src.demo.app --run-examples {OUT}
# PUSHED is printed only if the examples ran, and the commit and push worked
!git checkout -q -B rathi/R10-video && git add {OUT} && git commit -q -m "[R10] Demo example results ({VERDICT} verdict)" && git push -q -f origin rathi/R10-video && echo PUSHED

## 4. Launch the demo (click the `https://….gradio.live` link). This cell keeps running while the demo is open; stop it when you are done.

In [ ]:
!python -m src.demo.app --share